# Bayesian Regression

## Overview

Bayesian regression places prior distributions on regression coefficients and the error variance, producing a full posterior distribution over all parameters. This gives:
- Uncertainty-quantified coefficient estimates
- Posterior predictive distributions (not just point predictions)
- Natural regularisation through priors
- Principled handling of small samples

**Frequentist vs Bayesian regression:**

| Aspect | Frequentist OLS | Bayesian |
|---|---|---|
| Coefficients | Point estimates + SEs | Full posterior distributions |
| Predictions | Point + CI/PI | Posterior predictive distribution |
| Regularisation | Ridge/LASSO penalties | Shrinkage priors |
| Small n | Unreliable | Priors stabilise estimates |

---

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

rng = np.random.default_rng(42)
n = 60
elevation = rng.uniform(50, 400, n)
nitrate   = rng.gamma(2, 2, n)
treatment = rng.choice([0,1], n)
richness  = 25 - 0.03*elevation - 0.7*nitrate + 2.5*treatment + rng.normal(0, 3.5, n)
X = np.column_stack([np.ones(n), elevation, nitrate, treatment])
print(f"n={n}, p={X.shape[1]-1} predictors")

n=60, p=3 predictors


---
## Analytical Bayesian Linear Regression

In [3]:
# Normal-inverse-gamma conjugate: analytical posterior
# Prior: beta ~ N(b0, V0), sigma^2 ~ InvGamma(a0, b0)
# With flat/weakly informative prior (Zellner g-prior style)
g = n  # g-prior scaling
X_ols = sm.add_constant(pd.DataFrame({"elevation":elevation,"nitrate":nitrate,"treatment":treatment}))
ols = sm.OLS(richness, X_ols).fit()
print("OLS estimates (comparison):")
print(ols.summary().tables[1])
# Posterior under flat prior = OLS
print(f"\nUnder flat prior, Bayesian posterior mean = OLS estimate")
print(f"Bayesian interpretation: coefficients are random variables, not fixed unknowns")

OLS estimates (comparison):
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         22.4612      1.439     15.604      0.000      19.578      25.345
elevation     -0.0275      0.005     -5.671      0.000      -0.037      -0.018
nitrate       -0.3934      0.178     -2.204      0.032      -0.751      -0.036
treatment      3.8173      0.966      3.953      0.000       1.883       5.752

Under flat prior, Bayesian posterior mean = OLS estimate
Bayesian interpretation: coefficients are random variables, not fixed unknowns


---
## PyMC Bayesian Regression

In [4]:
try:
    import pymc as pm
    import arviz as az
    elev_s = (elevation - elevation.mean()) / elevation.std()
    nitr_s = (nitrate   - nitrate.mean())   / nitrate.std()
    with pm.Model() as reg_model:
        # Weakly informative priors
        intercept = pm.Normal("intercept", mu=20, sigma=10)
        b_elev    = pm.Normal("b_elevation", mu=0, sigma=5)
        b_nitr    = pm.Normal("b_nitrate",   mu=0, sigma=5)
        b_treat   = pm.Normal("b_treatment", mu=0, sigma=5)
        sigma     = pm.HalfNormal("sigma", sigma=10)
        mu_model  = (intercept + b_elev*elev_s + b_nitr*nitr_s
                     + b_treat*treatment)
        y_obs = pm.Normal("y_obs", mu=mu_model, sigma=sigma, observed=richness)
        trace = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                          target_accept=0.9, progressbar=False)
    print(az.summary(trace, var_names=["intercept","b_elevation","b_nitrate","b_treatment","sigma"]))
except ImportError:
    print("PyMC not installed: pip install pymc arviz")
    print("Bayesian regression with PyMC:")
    print("  - Priors placed on each coefficient")
    print("  - Full posterior sampled via NUTS")
    print("  - Posterior predictive checks validate model fit")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, b_elevation, b_nitrate, b_treatment, sigma]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 215 seconds.


               mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
intercept    14.739  0.633  13.566   15.939      0.007    0.007    7867.0   
b_elevation  -2.676  0.492  -3.606   -1.758      0.005    0.006    9496.0   
b_nitrate    -1.048  0.488  -1.957   -0.122      0.005    0.006    8767.0   
b_treatment   3.663  0.964   1.849    5.481      0.011    0.010    7714.0   
sigma         3.747  0.364   3.076    4.428      0.004    0.005    7721.0   

             ess_tail  r_hat  
intercept      5926.0    1.0  
b_elevation    5805.0    1.0  
b_nitrate      5231.0    1.0  
b_treatment    6317.0    1.0  
sigma          5524.0    1.0  


---
## Posterior Predictive Distribution

In [5]:
# Demonstrate posterior predictive via grid approximation (no PyMC dependency)
# Use analytical posterior under known sigma
sigma_known = 3.5
X_full = X.copy()
V0_inv = np.eye(4) * 0.04  # prior precision (weakly informative)
V0 = np.linalg.inv(V0_inv)
b0 = np.array([25, -0.03, -0.7, 2.5])
# Posterior
Vn_inv = V0_inv + (X_full.T @ X_full) / sigma_known**2
Vn = np.linalg.inv(Vn_inv)
bn = Vn @ (V0_inv @ b0 + (X_full.T @ richness) / sigma_known**2)
print(f"Posterior mean coefficients: {bn.round(3)}")
print(f"Posterior SD coefficients:   {np.sqrt(np.diag(Vn)).round(3)}")
# Posterior predictive for new sites
new_sites = np.array([[1,100,3,0],[1,200,2,1],[1,350,1,0]])
pred_mean = new_sites @ bn
pred_var  = [sigma_known**2 + x @ Vn @ x for x in new_sites]
print("\nPosterior predictive for new sites:")
for site, mu, var in zip(["low-elev ctrl","mid-elev rest","hi-elev ctrl"],
                          pred_mean, pred_var):
    ci = stats.norm.ppf([0.025,0.975], mu, np.sqrt(var))
    print(f"  {site:18s}: mean={mu:.2f}, 95% PI [{ci[0]:.2f},{ci[1]:.2f}]")

Posterior mean coefficients: [22.661 -0.028 -0.403  3.733]
Posterior SD coefficients:   [1.321 0.005 0.169 0.902]

Posterior predictive for new sites:
  low-elev ctrl     : mean=18.66, 95% PI [11.60,25.72]
  mid-elev rest     : mean=20.00, 95% PI [12.98,27.01]
  hi-elev ctrl      : mean=12.47, 95% PI [5.38,19.56]


---
## Shrinkage Priors: Regularisation the Bayesian Way

In [6]:
# Horseshoe and LASSO priors as Bayesian regularisation
# Demonstrate that tight priors shrink coefficients toward zero
noise_coefs = []
for prior_sd in [10.0, 2.0, 0.5, 0.1]:
    # Analytical posterior with normal prior
    Vn_inv_p = (np.eye(4)/prior_sd**2) + (X.T @ X)/sigma_known**2
    Vn_p     = np.linalg.inv(Vn_inv_p)
    bn_p     = Vn_p @ ((X.T @ richness)/sigma_known**2)
    noise_coefs.append(bn_p)
pd.set_option("display.float_format","{:.4f}".format)
coef_df = pd.DataFrame(noise_coefs,
                        columns=["intercept","elevation","nitrate","treatment"],
                        index=["prior_sd=10","prior_sd=2","prior_sd=0.5","prior_sd=0.1"])
print("Effect of prior width on posterior coefficients:")
print(coef_df)
print("\nNarrower priors -> stronger shrinkage toward zero (like Ridge regression)")

Effect of prior width on posterior coefficients:
              intercept  elevation  nitrate  treatment
prior_sd=10     22.0627    -0.0264  -0.3725     3.8851
prior_sd=2      15.6257    -0.0086  -0.0263     4.6217
prior_sd=0.5     3.1249     0.0309   0.6902     2.1673
prior_sd=0.1     0.1650     0.0500   0.2922     0.1316

Narrower priors -> stronger shrinkage toward zero (like Ridge regression)


---

## Common Pitfalls

**1. Using uninformative priors on all parameters without justification**  
Flat priors on regression coefficients can cause computational problems in MCMC and imply implausible parameter values (e.g. a slope of 1,000 is equally probable as 0.01). Weakly informative priors centred near zero (e.g. N(0,5) on standardised predictors) are almost always more defensible.

**2. Not standardising predictors before Bayesian regression**  
With unstandardised predictors, the natural scale of "weakly informative" differs enormously by variable (elevation in hundreds, nitrate in single digits). Standardise all continuous predictors so that N(0,5) is weakly informative across all coefficients.

**3. Reporting only posterior means and treating them as equivalent to OLS estimates**  
The point of Bayesian regression is the full posterior. Always report posterior SDs or HDIs alongside means, and plot the posterior distributions for scientifically important parameters.

**4. Not running posterior predictive checks**  
A Bayesian model can have good parameter estimates but still generate predictions that look nothing like the observed data. Always simulate from the posterior predictive distribution and compare to observed data visually.

**5. Confusing credible intervals on coefficients with prediction intervals for new observations**  
A credible interval on a coefficient captures uncertainty about the parameter. A posterior predictive interval for a new observation is wider — it includes both parameter uncertainty and residual variation. Both should be reported for complete uncertainty communication.

---
*python_methods_library - Samantha McGarrigle*